# Time Series exploration of Stock Market Data

## Key Questions we are asking;

1. What was the change in price of the stock over time?

2. What was the daily return of the stock on average?
3. What was the moving average of the various stocks?
4. What was the correlation between different stocks'?
5. How much value do we put at risk by investing in a particular stock?
6. How can we attempt to predict future stock behavior? (Predicting the closing price stock price of APPLE inc using LSTM)



In [2]:
# import libraries
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import yfinance as yf
from datetime import datetime, timedelta





In [13]:
# load data
# Stocks
tech_list = ['GOOGL', 'AAPL', 'MSFT', 'AMZN', 'META', 'TSLA', 'NVDA']
company_names = ['GOOGLE', 'APPLE', 'MICROSOFT', 'AMAZON', 'META', 'TESLA', 'NVIDIA']

# Dates
end = datetime.now()
start = datetime(end.year - 10, end.month, end.day)

# Store DataFrames
company_data = []

for ticker, name in zip(tech_list, company_names):
    df = yf.download(ticker, start=start, end=end)
    df['company_name'] = name
    df['Ticker'] = ticker
    company_data.append(df)
    
# Concatenate all company data into a single DataFrame
data = pd.concat(company_data, axis=0).reset_index()
data.head()




[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume,company_name,Ticker,Close,High,...,Close,High,Low,Open,Volume,Close,High,Low,Open,Volume
Ticker,,GOOGL,GOOGL,GOOGL,GOOGL,GOOGL,,,AAPL,AAPL,...,TSLA,TSLA,TSLA,TSLA,TSLA,NVDA,NVDA,NVDA,NVDA,NVDA
0,2016-03-21,37.795624,37.854138,37.255585,37.386503,28774000.0,GOOGLE,GOOGL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-03-22,37.690983,37.936455,37.583375,37.611145,22096000.0,GOOGLE,GOOGL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-03-23,37.567505,37.955798,37.491632,37.854633,24678000.0,GOOGLE,GOOGL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-03-24,37.432625,37.567508,37.217405,37.252117,31116000.0,GOOGLE,GOOGL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-03-28,37.355259,37.604201,37.293767,37.498571,21676000.0,GOOGLE,GOOGL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Let reshape the data for easier analysis
data1 = data.stack(level=1).rename_axis(['Date', 'Ticker']).reset_index(level=1)
data.head()


Price            Close                                                  \
Ticker            AAPL        AMZN       GOOGL        META        MSFT   
Date                                                                     
2025-03-20  213.170914  194.949997  162.237442  584.177429  383.901031   
2025-03-21  217.322815  196.210007  163.423340  594.395569  388.287415   
2025-03-24  219.772156  203.259995  167.100601  616.925293  390.093628   
2025-03-25  222.779022  205.710007  169.970627  624.362183  392.157776   
2025-03-26  220.568665  201.130005  164.489639  609.079773  387.007202   

Price                                     High                          ...  \
Ticker            NVDA        TSLA        AAPL        AMZN       GOOGL  ...   
Date                                                                    ...   
2025-03-20  118.502068  236.259995  216.546202  199.320007  164.320216  ...   
2025-03-21  117.672264  248.710007  217.890334  196.990005  163.672476  ...   
2025-03-24  121.381401  278.390015  220.518901  203.639999  167.738404  ...   
2025-03-25  120.661568  288.140015  223.127509  206.210007  170.040392  ...   
2025-03-26  113.733200  272.059998  224.043525  206.009995  169.023920  ...   

Price             Open                            Volume                      \
Ticker            MSFT        NVDA        TSLA      AAPL      AMZN     GOOGL   
Date                                                                           
2025-03-20  382.809383  116.522538  233.350006  48862900  38921100  28138500   
2025-03-21  380.308490  116.912449  234.990005  94127800  60056900  36625800   
2025-03-24  392.396009  119.851755  258.079987  44299500  41625400  30879100   
2025-03-25  390.927206  120.521601  283.600006  34493600  31171200  24174400   
2025-03-26  391.998985  118.702030  282.660004  34466100  32855300  28901600   

Price                                                 
Ticker          META      MSFT       NVDA       TSLA  
Date                                                  
2025-03-20  24336500  18470500  248829700   99028300  
2025-03-21  25015900  39675900  266498500  132728700  
2025-03-24  15741300  21004500  228452500  169079900  
2025-03-25  15312500  15775000  167447200  150361500  
2025-03-26  12609800  16108400  293463300  153629800  

[5 rows x 35 columns]

In [12]:

company_list = ['GOOGL', 'AAPL', 'MSFT', 'AMZN', 'META', 'TSLA', 'NVDA']
company_names = ['Google', 'Apple', 'Microsoft', 'Amazon', 'Meta', 'Tesla', 'NVIDIA']

dfs = []

for ticker, name in zip(company_list, company_names):
    df = yf.download(ticker, start='2020-01-01')
    df['company_name'] = name
    df['Ticker'] = ticker
    dfs.append(df)

# Combine all data
all_data = pd.concat(dfs, axis=0)

all_data.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,company_name,Ticker,Close,High,Low,...,Close,High,Low,Open,Volume,Close,High,Low,Open,Volume
Ticker,GOOGL,GOOGL,GOOGL,GOOGL,GOOGL,,,AAPL,AAPL,AAPL,...,TSLA,TSLA,TSLA,TSLA,TSLA,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,67.873016,67.873016,66.772614,66.867828,27278000.0,Google,GOOGL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,67.517967,68.124458,66.813788,66.847514,23408000.0,Google,GOOGL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-06,69.317596,69.342885,66.996282,67.027518,46768000.0,Google,GOOGL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-07,69.183701,69.599764,69.007658,69.449010,34330000.0,Google,GOOGL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-08,69.676132,70.013841,69.060720,69.169319,35314000.0,Google,GOOGL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
